# Exercise 2: Function Calling & Tools

## Learning Objectives

In this exercise, you will:
- Understand the difference between tools (function calling) and RAG
- Implement search_flights and search_hotels functions
- Add tools to your agent
- Handle errors gracefully with the error-in-context pattern
- Filter results by budget constraints

**Estimated time:** 30 minutes

## What is Function Calling?

**⏱️ Estimated time: ~5 minutes**

<!-- INSTRUCTOR: This is THE key concept distinction - spend time here -->

Function calling (also called "tool use") lets your agent **take actions** by calling Python functions you provide. This is how agents access real-time data, perform calculations, or interact with external systems.

---

### Tools vs RAG: The Decision Framework

**This is critical!** You need to know when to use function calling vs RAG (Retrieval-Augmented Generation).

| Use Case | Tool (Function Calling) | RAG (Knowledge Retrieval) |
|----------|-------------------------|---------------------------|
| **When to use** | Data changes frequently (real-time) | Data is static or slow-changing |
| **Examples** | Flight availability<br>Hotel pricing<br>Current weather<br>Stock prices | Destination guides<br>Visa requirements<br>Cultural tips<br>Packing lists |
| **Key question** | "Does this data change while user is talking?" | "Is this reference information?" |

**In our travel agent:**
- ✅ **Tools** → search_flights(), search_hotels() (prices/availability change constantly)
- ✅ **RAG** → destination guides, travel tips (we'll add this in Exercise 3)

> 💡 **The Key Question:** "Does this data change while the user is talking to the agent?"
> - **YES** → Use a tool (function calling)
> - **NO** → Use RAG (knowledge retrieval)

---

### How Function Calling Works in ADK

```
User: "Find flights from San Francisco to Tokyo"
        ↓
Agent: Decides to call search_flights()
        ↓
Your Function: Returns flight data
        ↓
Agent: Formats results for user
        ↓
User sees: "Here are 3 available flights..."
```

**The magic:** ADK automatically converts your Python function into a tool schema that the LLM understands!

## Setup Instructions

**⏱️ Estimated time: ~2 minutes**

This exercise assumes you have completed Exercise 1 and your environment is ready.

If you haven't completed setup yet, please go back to: [00-setup-verification.ipynb](00-setup-verification.ipynb)

In [ ]:
# @title Quick Setup (Run this cell first)
# This cell sets up your environment - you can collapse it after running

import os
import google.generativeai as genai
from getpass import getpass

# Step 1: Install Google ADK
!pip install -q google-adk==1.23.0
print("✓ google-adk 1.23.0 installed")

# Step 2: Configure API Key
api_key = getpass('Enter your Google AI API Key: ')
genai.configure(api_key=api_key)
os.environ['GOOGLE_API_KEY'] = api_key
print("✓ API Key configured")

print("\n🚀 Ready to build agents with tools!")

In [ ]:
# Import ADK components for running agents
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

# Create session service for conversation management
session_service = InMemorySessionService()
user_id = "workshop_participant"

print("✓ ADK components imported")
print("✓ Session service created")

---

## ✏️ Exercise 2A: Implement search_flights

**⏱️ Estimated time: ~7 minutes**

<!-- INSTRUCTOR: This is the first function they write - walk through docstring importance -->

Complete the `search_flights` function below. This is a **mock API** - it returns sample data instead of calling a real travel API (avoiding costs and rate limits during the workshop).

### Your Tasks:

1. **Fill in the function signature** with type hints for parameters
2. **Write a descriptive docstring** with Args and Returns sections
3. **Return mock flight data** in the specified structure

### Why This Matters:

- **Type hints** → ADK uses these to generate the tool schema automatically
- **Docstring** → The LLM reads this to decide when and how to use your tool
- **Good descriptions** → Better tool selection by the agent

### Hints:

- Use type hints like: `origin: str, destination: str`
- Include **example values** in docstring: `'SFO'`, `'NRT'`, `'2026-03-15'`
- Return a dict with keys: `status`, `flights`, `currency`, `route`, `date`

In [ ]:
from datetime import datetime
from typing import Optional

def search_flights(
    # TODO: Add parameters with type hints
    # origin: str - Airport code like 'SFO'
    # destination: str - Airport code like 'NRT'
    # departure_date: str - Date in YYYY-MM-DD format
    # passengers: int = 1
) -> dict:
    """
    TODO: Write docstring with Args and Returns sections
    
    The LLM reads this docstring to understand your tool!
    
    Example format:
    Search for available flights between airports.
    
    Args:
        origin: Departure airport code (e.g., 'SFO', 'LAX', 'JFK')
        destination: Arrival airport code (e.g., 'NRT', 'CDG', 'LHR')
        ...
    
    Returns:
        Dictionary with 'status', 'flights' list, 'currency', 'route', 'date'
    """
    # Debug output - shows when function is called
    print(f"🔧 search_flights called: {origin} -> {destination}")
    
    # TODO: Return mock flight data
    # Return structure:
    # {
    #     "status": "success",
    #     "flights": [
    #         {
    #             "airline": "United Airlines",
    #             "flight_number": "UA837",
    #             "price": 850,
    #             "departure": "08:30",
    #             "arrival": "14:30+1",
    #             "duration": "11h 00m",
    #             "aircraft": "Boeing 787-9"
    #         },
    #         # Add 2-3 more flight options with different prices
    #     ],
    #     "currency": "USD",
    #     "route": f"{origin} → {destination}",
    #     "date": departure_date,
    # }
    pass

### ✅ Checkpoint: Test your function

Run this cell to test if your function works correctly.

**Expected output:**
- Dictionary with `"status": "success"`
- A `flights` list with 2-3 flight options
- Each flight has airline, price, times, etc.

In [ ]:
# Test the function
result = search_flights("SFO", "NRT", "2026-03-15", passengers=2)
print(result)

# You should see:
# - status: "success"
# - flights: list with flight data
# - currency, route, date

**If you got an error, try these troubleshooting steps:**

❌ **NameError: name 'origin' is not defined**
- You need to add the parameters to the function signature
- Make sure you uncommented the TODO lines and added proper type hints

❌ **Function returns None**
- You need to replace `pass` with a `return` statement
- Return a dictionary with the structure shown in the comments

❌ **No debug output**
- The `print(f"🔧 search_flights called...")` line should execute
- Make sure your function signature includes the `origin` and `destination` parameters

---

## ✏️ Exercise 2B: Implement search_hotels

**⏱️ Estimated time: ~7 minutes**

Now implement a hotel search function following the same pattern.

### Your Tasks:

1. **Add parameters:** location (str), check_in (str), check_out (str), guests (int = 1)
2. **Write docstring** with examples
3. **Return mock hotel data** with 2-3 hotel options

### Hotel Data Structure:

```python
{
    "status": "success",
    "hotels": [
        {
            "name": "Park Hyatt Tokyo",
            "stars": 5,
            "price_per_night": 450,
            "rating": 4.8,
            "amenities": ["Pool", "Spa", "Restaurant"]
        },
        # ... more hotels
    ],
    "location": location,
    "check_in": check_in,
    "check_out": check_out,
    "currency": "USD"
}
```

In [ ]:
def search_hotels(
    # TODO: Add parameters with type hints
    # location: str - City name like 'Tokyo' or 'Paris'
    # check_in: str - Date in YYYY-MM-DD format
    # check_out: str - Date in YYYY-MM-DD format
    # guests: int = 1
) -> dict:
    """
    TODO: Write docstring
    
    Search for available hotels in a destination.
    
    Args:
        location: City or area name (e.g., 'Tokyo', 'Paris', 'New York')
        ...
    
    Returns:
        Dictionary with 'status', 'hotels' list, location, dates, currency
    """
    # Debug output
    print(f"🔧 search_hotels called: {location}, {check_in} to {check_out}")
    
    # TODO: Return mock hotel data
    # Include 2-3 hotels with varied prices (e.g., $180, $220, $450)
    pass

### ✅ Checkpoint: Test your hotel search

In [ ]:
# Test the function
result = search_hotels("Tokyo", "2026-03-15", "2026-03-20", guests=2)
print(result)

# Expected: Dictionary with hotel data

---

## ✏️ Exercise 2C: Error Handling (Error-in-Context Pattern)

**⏱️ Estimated time: ~8 minutes**

<!-- INSTRUCTOR: This is the CRITICAL pattern - emphasize why exceptions don't work -->

Real tools need to handle errors gracefully. But here's the key insight:

### ❌ DON'T Do This:

```python
def bad_search(date: str):
    if invalid_date:
        raise ValueError("Invalid date")  # ❌ LLM NEVER SEES THIS!
```

**Problem:** When you raise an exception, the LLM doesn't see the error message. The agent can't learn from it or help the user fix the problem.

### ✅ DO This Instead (Error-in-Context):

```python
def good_search(date: str):
    if invalid_date:
        return {  # ✅ LLM CAN SEE AND RESPOND TO THIS!
            "status": "error",
            "error_message": "Invalid date format. Use YYYY-MM-DD (e.g., 2026-03-15)",
            "example": "2026-03-15"
        }
```

**Why this works:** The LLM sees the error message and can help the user correct their request!

---

### Your Task:

Modify your `search_flights` function to validate the date format and return an error dict if invalid.

In [ ]:
# Enhanced search_flights with error handling
def search_flights(
    origin: str,
    destination: str,
    departure_date: str,
    passengers: int = 1
) -> dict:
    """
    Search for available flights between airports.
    
    Args:
        origin: Departure airport code (e.g., 'SFO', 'LAX', 'JFK')
        destination: Arrival airport code (e.g., 'NRT', 'CDG', 'LHR')
        departure_date: Departure date in YYYY-MM-DD format (e.g., '2026-03-15')
        passengers: Number of passengers (default 1)
    
    Returns:
        Dictionary with flight data or error information
    """
    print(f"🔧 search_flights called: {origin} -> {destination}, {departure_date}")
    
    # TODO: Add date validation
    # Try to parse the date with datetime.strptime(departure_date, '%Y-%m-%d')
    # If it raises ValueError, return error dict:
    # {
    #     "status": "error",
    #     "error_message": f"Invalid date format: '{departure_date}'. Please use YYYY-MM-DD (e.g., '2026-03-15').",
    #     "requested_date": departure_date,
    #     "example": "2026-03-15"
    # }
    
    # If validation passes, return success data
    return {
        "status": "success",
        "flights": [
            {
                "airline": "United Airlines",
                "flight_number": "UA837",
                "price": 850,
                "departure": "08:30",
                "arrival": "14:30+1",
                "duration": "11h 00m",
                "aircraft": "Boeing 787-9"
            },
            {
                "airline": "ANA",
                "flight_number": "NH7",
                "price": 920,
                "departure": "11:00",
                "arrival": "17:00+1",
                "duration": "11h 00m",
                "aircraft": "Boeing 777-300ER"
            },
            {
                "airline": "JAL",
                "flight_number": "JL2",
                "price": 890,
                "departure": "13:45",
                "arrival": "19:45+1",
                "duration": "11h 00m",
                "aircraft": "Boeing 787-8"
            },
        ],
        "currency": "USD",
        "route": f"{origin} → {destination}",
        "date": departure_date,
    }

### ✅ Checkpoint: Test error handling

In [ ]:
# Test with VALID date
print("Valid date:")
result = search_flights("SFO", "NRT", "2026-03-15")
print(f"Status: {result['status']}\n")

# Test with INVALID date
print("Invalid date:")
result = search_flights("SFO", "NRT", "March 15")
print(f"Status: {result['status']}")
if result['status'] == 'error':
    print(f"Error: {result['error_message']}")

**Expected:**
- First test: `status: "success"`
- Second test: `status: "error"` with helpful error message

In [ ]:
# Create agent with tools
from google.adk.agents import Agent

agent = Agent(
    model='gemini-2.5-flash',
    name='travel_booking_assistant',
    description='Travel booking assistant with flight and hotel search',
    
    instruction='''You are a travel booking assistant.

YOUR TOOLS:
- search_flights(): Find available flights between airports
- search_hotels(): Find accommodation in destinations

HOW TO HELP USERS:
1. Gather key details: destination, dates, number of travelers, budget
2. Use your tools to find real options
3. Present 2-3 best matches with prices and details
4. If a tool returns an error, help the user fix their request

IMPORTANT:
- When calling search_flights, use airport codes (SFO, NRT, LAX, CDG, etc.)
- When calling search_hotels, use city names (Tokyo, Paris, New York, etc.)
- Dates must be in YYYY-MM-DD format (e.g., 2026-03-15)
- If user mentions budget, filter results to show only affordable options

Be friendly, concise, and helpful!''',
    
    tools=[
        # TODO: Add your functions here
        # search_flights,
        # search_hotels,
    ],
)

print("✅ Agent created with tools!")

In [ ]:
# Test function calling with proper Runner pattern
async def test_agent_with_tools(agent, queries):
    """Run queries against agent using Runner and Sessions"""
    # Create ONE session for this conversation
    session = await session_service.create_session(
        app_name=agent.name,
        user_id=user_id
    )
    
    # Create runner
    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )
    
    # Run each query
    for query in queries:
        print(f"\n{'='*60}")
        print(f"🧑 You: {query}")
        print(f"{'='*60}")
        
        final_response = ""
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            # Watch for function calls (tool invocations)
            if hasattr(event.content, 'parts'):
                for part in event.content.parts:
                    if hasattr(part, 'function_call') and part.function_call:
                        print(f"🔧 Tool called: {part.function_call.name}")
            
            if event.is_final_response():
                final_response = event.content.parts[0].text
                break
        
        print(f"\n🤖 Agent: {final_response}")

# Test queries
test_queries = [
    "Find me flights from San Francisco to Tokyo on March 15",
    "Now find a hotel in Tokyo for March 15-20",
]

# Use await instead of asyncio.run - Colab has a running event loop
await test_agent_with_tools(agent, test_queries)

### What Just Happened?

Did you see the debug output? Lines like:
```
🔧 search_flights called: SFO -> NRT, 2026-03-15
🔧 search_hotels called: Tokyo, 2026-03-15 to 2026-03-20
```

This proves your agent is:
1. **Understanding** the user's intent
2. **Deciding** which tool to call
3. **Extracting** the right parameters (airport codes, dates)
4. **Calling** your Python functions
5. **Formatting** the results for the user

**All automatically!** You just provided the tools and instruction.

### ✅ Checkpoint: Troubleshooting

**If the agent didn't call your tools:**

❌ **No debug output / Agent says "I can't help with that"**
- Make sure you uncommented the tools in the `tools=[...]` list
- Check that your functions have proper docstrings
- Re-run the agent creation cell

❌ **Agent called the wrong tool**
- Check your docstrings - they should clearly describe what each tool does
- Make function names descriptive: `search_flights` not just `search`

❌ **Date format error**
- The agent should convert "March 15" to "2026-03-15" automatically
- If not, your instruction might need to be more explicit about date formats

---

## ✏️ Exercise 2F: Budget Filtering

**⏱️ Estimated time: ~5 minutes**

Let's add budget awareness! Modify your `search_flights` function to accept an optional `max_price` parameter.

In [ ]:
# Enhanced search_flights with budget filtering
def search_flights(
    origin: str,
    destination: str,
    departure_date: str,
    passengers: int = 1,
    max_price: Optional[int] = None  # NEW: Budget constraint
) -> dict:
    """
    Search for available flights between airports.
    
    Args:
        origin: Departure airport code (e.g., 'SFO', 'LAX', 'JFK')
        destination: Arrival airport code (e.g., 'NRT', 'CDG', 'LHR')
        departure_date: Departure date in YYYY-MM-DD format (e.g., '2026-03-15')
        passengers: Number of passengers (default 1)
        max_price: Maximum price per person in USD (optional)
    
    Returns:
        Dictionary with flight data or error information
    """
    print(f"🔧 search_flights called: {origin} -> {destination}, {departure_date}")
    if max_price:
        print(f"   Budget filter: max ${max_price} per person")
    
    # Validate date
    try:
        datetime.strptime(departure_date, '%Y-%m-%d')
    except ValueError:
        return {
            "status": "error",
            "error_message": f"Invalid date format: '{departure_date}'. Please use YYYY-MM-DD (e.g., '2026-03-15').",
            "requested_date": departure_date,
            "example": "2026-03-15"
        }
    
    # All available flights
    all_flights = [
        {
            "airline": "United Airlines",
            "flight_number": "UA837",
            "price": 850,
            "departure": "08:30",
            "arrival": "14:30+1",
            "duration": "11h 00m",
            "aircraft": "Boeing 787-9"
        },
        {
            "airline": "ANA",
            "flight_number": "NH7",
            "price": 920,
            "departure": "11:00",
            "arrival": "17:00+1",
            "duration": "11h 00m",
            "aircraft": "Boeing 777-300ER"
        },
        {
            "airline": "JAL",
            "flight_number": "JL2",
            "price": 890,
            "departure": "13:45",
            "arrival": "19:45+1",
            "duration": "11h 00m",
            "aircraft": "Boeing 787-8"
        },
    ]
    
    # TODO: Filter flights by budget if max_price is provided
    # flights = [f for f in all_flights if f["price"] <= max_price] if max_price else all_flights
    #
    # If no flights match budget, return error:
    # if max_price and not flights:
    #     return {
    #         "status": "error",
    #         "error_message": f"No flights found under ${max_price}. Lowest available: ${min(f['price'] for f in all_flights)}",
    #         "max_price": max_price
    #     }
    
    flights = all_flights  # Replace this with filtered flights
    
    return {
        "status": "success",
        "flights": flights,
        "currency": "USD",
        "route": f"{origin} → {destination}",
        "date": departure_date,
    }

In [ ]:
# Recreate agent with updated budget-aware instruction
from google.adk.agents import Agent

agent = Agent(
    model='gemini-2.5-flash',
    name='travel_booking_assistant',
    description='Travel booking assistant with flight and hotel search',
    instruction='''You are a travel booking assistant.

YOUR TOOLS:
- search_flights(): Find available flights between airports
- search_hotels(): Find accommodation in destinations

HOW TO HELP USERS:
1. Gather key details: destination, dates, number of travelers, budget
2. Use your tools with budget constraints when mentioned
3. Present only options within their budget
4. If no options fit budget, explain and suggest alternatives

IMPORTANT:
- When user mentions budget ("under $900", "$800 max"), use max_price parameter
- Only show flights that fit their budget
- If all options exceed budget, suggest how to adjust (different dates, routes)

Be friendly and budget-conscious!''',
    tools=[search_flights, search_hotels],
)

print("✅ Agent created with budget-aware instruction!")

# Test budget filtering using proper Runner pattern
async def test_budget_filtering():
    """Test budget-aware search with proper ADK pattern"""
    # Create session
    session = await session_service.create_session(
        app_name=agent.name,
        user_id=user_id
    )
    
    # Create runner
    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )
    
    # Test 1: Low budget
    print("Test 1: Budget $800")
    print("="*60)
    
    final_response = ""
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=Content(parts=[Part(text="Find me flights from SFO to Tokyo on March 15, 2026. My budget is $800 per person.")], role="user")
    ):
        if event.is_final_response():
            final_response = event.content.parts[0].text
            break
    
    print(f"Agent: {final_response}\n")
    
    # Test 2: Higher budget (same session - agent remembers context!)
    print("\n" + "="*60)
    print("Test 2: Budget $900")
    print("="*60)
    
    final_response = ""
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,  # Same session!
        new_message=Content(parts=[Part(text="Now try with a budget of $900 per person.")], role="user")
    ):
        if event.is_final_response():
            final_response = event.content.parts[0].text
            break
    
    print(f"Agent: {final_response}")

# Run the test (use await instead of asyncio.run - Colab has running event loop)
await test_budget_filtering()

---

## Challenge (Optional)

Ready for more practice? Try these challenges:

### Challenge 1: Add Date Validation

Enhance `search_flights` to check if the departure date is in the future. Return an error if it's in the past.

### Challenge 2: Add Hotel Budget Filtering

Add a `max_price_per_night` parameter to `search_hotels` and filter results.

### Challenge 3: Multi-City Search

Test a complex query: "I want to fly from San Francisco to Tokyo on March 15, then find a hotel for 5 nights under $200 per night."

Watch your agent call both tools in sequence!

### Challenge 4: Error Recovery

Try: "Find me flights from SFO to Tokyo on March 32"

The agent should see the date error and help you correct it.

In [ ]:
# @title Solutions: Complete Implementations (Expand to see)

# Solution for search_flights with full validation
def search_flights_solution(
    origin: str,
    destination: str,
    departure_date: str,
    passengers: int = 1,
    max_price: Optional[int] = None
) -> dict:
    """
    Search for available flights between airports.
    
    Args:
        origin: Departure airport code (e.g., 'SFO', 'LAX', 'JFK')
        destination: Arrival airport code (e.g., 'NRT', 'CDG', 'LHR')
        departure_date: Departure date in YYYY-MM-DD format (e.g., '2026-03-15')
        passengers: Number of passengers (default 1)
        max_price: Maximum price per person in USD (optional)
    
    Returns:
        Dictionary with flight data or error information
    """
    print(f"🔧 search_flights called: {origin} -> {destination}, {departure_date}")
    
    # Validate date format
    try:
        departure_dt = datetime.strptime(departure_date, '%Y-%m-%d')
    except ValueError:
        return {
            "status": "error",
            "error_message": f"Invalid date format: '{departure_date}'. Please use YYYY-MM-DD (e.g., '2026-03-15').",
            "requested_date": departure_date,
            "example": "2026-03-15"
        }
    
    # Validate date is in future
    if departure_dt.date() < datetime.now().date():
        return {
            "status": "error",
            "error_message": f"Departure date {departure_date} is in the past. Please provide a future date.",
            "requested_date": departure_date
        }
    
    # Mock flight data
    all_flights = [
        {
            "airline": "United Airlines",
            "flight_number": "UA837",
            "price": 850,
            "departure": "08:30",
            "arrival": "14:30+1",
            "duration": "11h 00m",
            "aircraft": "Boeing 787-9"
        },
        {
            "airline": "ANA",
            "flight_number": "NH7",
            "price": 920,
            "departure": "11:00",
            "arrival": "17:00+1",
            "duration": "11h 00m",
            "aircraft": "Boeing 777-300ER"
        },
        {
            "airline": "JAL",
            "flight_number": "JL2",
            "price": 890,
            "departure": "13:45",
            "arrival": "19:45+1",
            "duration": "11h 00m",
            "aircraft": "Boeing 787-8"
        },
    ]
    
    # Filter by budget
    if max_price:
        flights = [f for f in all_flights if f["price"] <= max_price]
        if not flights:
            lowest = min(f["price"] for f in all_flights)
            return {
                "status": "error",
                "error_message": f"No flights found under ${max_price}. Lowest available price: ${lowest}",
                "max_price": max_price,
                "lowest_price": lowest
            }
    else:
        flights = all_flights
    
    return {
        "status": "success",
        "flights": flights,
        "currency": "USD",
        "route": f"{origin} → {destination}",
        "date": departure_date,
        "passengers": passengers
    }

# Solution for search_hotels
def search_hotels_solution(
    location: str,
    check_in: str,
    check_out: str,
    guests: int = 1,
    max_price_per_night: Optional[int] = None
) -> dict:
    """
    Search for available hotels in a destination.
    
    Args:
        location: City or area name (e.g., 'Tokyo', 'Paris', 'New York')
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
        guests: Number of guests (default 1)
        max_price_per_night: Maximum price per night in USD (optional)
    
    Returns:
        Dictionary with hotel data or error information
    """
    print(f"🔧 search_hotels called: {location}, {check_in} to {check_out}")
    
    # Validate dates
    try:
        checkin_dt = datetime.strptime(check_in, '%Y-%m-%d')
        checkout_dt = datetime.strptime(check_out, '%Y-%m-%d')
    except ValueError:
        return {
            "status": "error",
            "error_message": "Invalid date format. Use YYYY-MM-DD.",
            "check_in": check_in,
            "check_out": check_out
        }
    
    # Validate date order
    if checkout_dt <= checkin_dt:
        return {
            "status": "error",
            "error_message": f"Check-out date must be after check-in date.",
            "check_in": check_in,
            "check_out": check_out
        }
    
    # Mock hotel data
    all_hotels = [
        {
            "name": "Park Hyatt Tokyo",
            "stars": 5,
            "price_per_night": 450,
            "rating": 4.8,
            "amenities": ["Pool", "Spa", "Restaurant", "Gym"]
        },
        {
            "name": "Shinjuku Granbell Hotel",
            "stars": 4,
            "price_per_night": 180,
            "rating": 4.5,
            "amenities": ["Restaurant", "Bar", "Gym"]
        },
        {
            "name": "MUJI Hotel Ginza",
            "stars": 4,
            "price_per_night": 220,
            "rating": 4.6,
            "amenities": ["Restaurant", "Minimalist design"]
        },
    ]
    
    # Filter by budget
    if max_price_per_night:
        hotels = [h for h in all_hotels if h["price_per_night"] <= max_price_per_night]
        if not hotels:
            lowest = min(h["price_per_night"] for h in all_hotels)
            return {
                "status": "error",
                "error_message": f"No hotels found under ${max_price_per_night}/night. Lowest available: ${lowest}/night",
                "max_price_per_night": max_price_per_night
            }
    else:
        hotels = all_hotels
    
    return {
        "status": "success",
        "hotels": hotels,
        "location": location,
        "check_in": check_in,
        "check_out": check_out,
        "currency": "USD"
    }

---

## What's Next?

🎉 **Congratulations!** You've built an agent with real-time data access!

Your agent can now:
- ✅ Search flights and hotels
- ✅ Handle errors gracefully
- ✅ Filter by budget

But when you ask "What's the best time to visit Japan?", your agent can't answer from its knowledge.

**Coming up in Exercise 3: RAG (Retrieval-Augmented Generation)**
- Build a destination knowledge base
- Add travel guides and tips
- Combine real-time tools with static knowledge
- Create a truly comprehensive travel assistant!

Ready to continue? Open [03-rag-knowledge.ipynb](03-rag-knowledge.ipynb)